In [5]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="h5py")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import re
import os
from pathlib import Path
import shutil
from functools import partial
import numpy as np
from glob import glob
from torchinfo import summary

import torch
import torch.nn as nn
import torch.optim as optim

# Import Custom Modules
from dataloader import FaciesDataset
#from utils import parse_hybrid_args, validate_dataset, save_config

# VoxGAN Imports
from voxgan.data.datasets import *
from voxgan.models.base import CustomGAN
from voxgan.networks import resnet
from voxgan.networks.utils import initialize_weights_normal
from voxgan.models.loss import R1Regularization
from voxgan.models.metrics import MSSWD, LoS
run_dir = os.getcwd()


In [8]:
nz = 100
nl = (3, 5, 5)
nc=9

In [9]:
# Strictly matching working script: Always use Tanh for output bounds
last_activation = nn.Tanh

generator = partial(resnet.DeepGenerator3d,
                        nz=nz, ngf=64, nc=nc, nl=nl,
                        max_factor=16, residual_weight=1., mode='nearest', kernel_size=3,
                        layer_normalization=nn.BatchNorm3d,
                        last_layer_normalization=nn.BatchNorm3d,
                        weight_normalization=nn.utils.parametrizations.spectral_norm,
                        activation=partial(nn.LeakyReLU, negative_slope=0.2, inplace=True),
                        last_activation=last_activation,
                        use_double_conv=False, use_double_resblocks=False,
                        use_attention=False, skip_z=False, split_z=False)

discriminator = partial(resnet.DeepDiscriminator3d,
                            ndf=64, nc=nc, nl=nl,
                            max_factor=16, residual_weight=1., kernel_size=3,
                            layer_normalization=None,
                            weight_normalization=nn.utils.parametrizations.spectral_norm,
                            activation=partial(nn.LeakyReLU, negative_slope=0.2, inplace=True),
                            use_double_conv=False, use_double_resblocks=False,                      
                            use_attention=False)
    
gen_model = generator()
disc_model = discriminator()

# 2. Print summaries to console
print("\n" + "="*50)
print("GENERATOR ARCHITECTURE:")
# Input size is explicitly 5D -> (Batch, Latent/Channels, Depth, Height, Width)
gen_summary = summary(gen_model, input_size=(1, 100, 1, 1, 1), verbose=1)

print("\n" + "="*50)
print("DISCRIMINATOR ARCHITECTURE:")
# FIX: Updated the input spatial dimensions to perfectly match your ZYX dataset (32, 128, 128)
disc_summary = summary(disc_model, input_size=(1, nc, 32, 128, 128), verbose=1)


GENERATOR ARCHITECTURE:
Layer (type:depth-idx)                                       Output Shape              Param #
DeepGenerator3d                                              [1, 9, 32, 128, 128]      --
├─BlockSequential: 1-1                                       [1, 9, 32, 128, 128]      --
│    └─ConvBlockTranspose3d: 2-1                             [1, 1024, 4, 4, 4]        --
│    │    └─BlockSequential: 3-1                             [1, 1024, 4, 4, 4]        6,553,600
│    └─DeepResBlockUpscale3d: 2-2                            [1, 1024, 8, 8, 8]        --
│    │    └─BlockSequential: 3-2                             [1, 1024, 8, 8, 8]        2,296,832
│    │    └─Upsample: 3-3                                    [1, 1024, 8, 8, 8]        --
│    └─DeepResBlockUpscale3d: 2-3                            [1, 512, 16, 16, 16]      --
│    │    └─BlockSequential: 3-4                             [1, 512, 16, 16, 16]      2,165,760
│    │    └─Upsample: 3-5                        

In [15]:
from torchviz import make_dot
import torch
import os
run_dir = os.getcwd()

# 1. Instantiate the models
gen_model = generator()
disc_model = discriminator()

# 2. Create dummy inputs
dummy_z = torch.randn(1, 100, 1, 1, 1) 
dummy_img = torch.randn(1, 9, 32, 128, 128) 

# 3. Pass inputs through the models
gen_output = gen_model(dummy_z)
disc_output = disc_model(dummy_img)

# --- THE FIX: Extract the actual tensor if the model returns a dictionary ---
if isinstance(gen_output, dict):
    gen_tensor = list(gen_output.values())[0]
elif isinstance(gen_output, (list, tuple)):
    gen_tensor = gen_output[0]
else:
    gen_tensor = gen_output

if isinstance(disc_output, dict):
    disc_tensor = list(disc_output.values())[0]
elif isinstance(disc_output, (list, tuple)):
    disc_tensor = disc_output[0]
else:
    disc_tensor = disc_output
# --------------------------------------------------------------------------

# 4. Generate and save the plots
print("\nGenerating visual computational graphs. This might take a few seconds...")

gen_dot = make_dot(gen_tensor, params=dict(gen_model.named_parameters()))
gen_dot.format = 'png'
gen_dot.render(os.path.join(run_dir, "Generator_Graph"))

disc_dot = make_dot(disc_tensor, params=dict(disc_model.named_parameters()))
disc_dot.format = 'png'
disc_dot.render(os.path.join(run_dir, "Discriminator_Graph"))

print(f"Saved Generator_Graph.png and Discriminator_Graph.png to: {run_dir}\n")


Generating visual computational graphs. This might take a few seconds...
Saved Generator_Graph.png and Discriminator_Graph.png to: c:\Users\mathi\Desktop\TU Delft\TU Delft year 5\Master Thesis\Thesis-project-DGM\scripts\gan_pipeline\02_training



In [7]:
from torchview import draw_graph
import os
import torch
from torchviz import make_dot
import torch
import os

# 1. Instantiate the models
gen_model = generator()
disc_model = discriminator()

# 2. Draw and save the Generator
# 'depth=2' keeps the graph grouped in clean ResNet blocks instead of messy math nodes
draw_graph(gen_model, input_size=(1, 100, 1, 1, 1), 
           expand_nested=True, depth=2, save_graph=True, 
           filename=os.path.join(run_dir, "Generator_Layer_Graph_single"))

# 3. Draw and save the Discriminator
draw_graph(disc_model, input_size=(1, 1, 32, 128, 128), 
           expand_nested=True, depth=2, save_graph=True, 
           filename=os.path.join(run_dir, "Discriminator_Layer_Graph_single"))

print(f"Saved beautiful layer graphs to: {run_dir}")

couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.

Saved beautiful layer graphs to: c:\Users\mathi\Desktop\TU Delft\TU Delft year 5\Master Thesis\Thesis-project-DGM\scripts\gan_pipeline\02_training
